# 

# Define Libraries

In [15]:
import pandas as pd
import numpy as np
import duckdb as dd

## Check for the duckdb table path

In [16]:
import os
from pathlib import Path
print(os.getcwd())
print(list(Path(r"d:\Model_Monitoring_Individual_scripts").rglob("dd_ml_models_vector_details.duckdb")))

d:\Model_Monitoring_Individual_scripts\Common_Query
[WindowsPath('d:/Model_Monitoring_Individual_scripts/dd_ml_models_vector_details.duckdb')]


## Add the path to the ML_vector_analysis function

In [17]:
import sys
sys.path.append(r"D:\Model_Monitoring_Individual_scripts\Common_Query")
import Ml_vector_analysis as a

In [18]:
con = a.connect(r"D:\Model_Monitoring_Individual_scripts\dd_ml_models_vector_details.duckdb")   # read-only
checks = a.sanity_checks(con)
for name, df in checks.items():
    print(name); print(df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

schema
             column_name column_type null   key default extra
0             customerId      BIGINT  YES  None    None  None
1   digitalLoanAccountId     VARCHAR  YES  None    None  None
2       modelDisplayName     VARCHAR  YES  None    None  None
3         modelVersionId     VARCHAR  YES  None    None  None
4         trenchCategory     VARCHAR  YES  None    None  None
5               deviceOs     VARCHAR  YES  None    None  None
6          new_loan_type     VARCHAR  YES  None    None  None
7       application_date        DATE  YES  None    None  None
8         Data_selection     VARCHAR  YES  None    None  None
9               Features     VARCHAR  YES  None    None  None
10                Values     VARCHAR  YES  None    None  None
11             Data_type     VARCHAR  YES  None    None  None
12             is_delete     BOOLEAN  YES  None    None  None
13             is_active     BOOLEAN  YES  None    None  None
14              run_date        DATE  YES  None    None  None
v

In [19]:
con.execute("""SELECT distinct modelDisplayName, modelVersionId, Data_Selection FROM dd_ml_models_vector_details
              order by 1,2, 3""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,modelDisplayName,modelVersionId,Data_selection
0,Transaction_Score_Cash_Model,v1,Dev_Test
1,Transaction_Score_Cash_Model,v1,Dev_Train
2,alpha_income_model,v1,Prod
3,alpha_stack_model_cash,v1,Dev_Test
4,alpha_stack_model_cash,v1,Dev_Train
...,...,...,...
105,device_price_model_cash,v2(shadow),Prod
106,gamma_demo_model_cash,v1,Prod
107,gamma_stack_model_cash,v1,Prod
108,score_quick_new_2,v2,Prod


In [20]:
output_file = r"D:\Model_Monitoring_Individual_scripts\dd_ml_models_vector_details.csv"   # Windows
# output_file = "/tmp/dd_ml_models_vector_details.csv"     # Linux

con.execute(f"""
COPY (
    SELECT DISTINCT
        modelDisplayName,
        modelVersionId,
        Data_Selection, 
        count(distinct customerId) cnt_dstct_customer,
        count(distinct digitalLoanAccountId) cnt_dstct_loans,
        min(application_date) firstdate,
        max(application_date) lastdate
    FROM dd_ml_models_vector_details
    group by 1,2,3
    ORDER BY 1,2,3
) TO '{output_file}'
WITH (HEADER, DELIMITER ',');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [21]:
output_file = r"D:\Model_Monitoring_Individual_scripts\dd_ml_models_vector_features_details.csv"   # Windows
# output_file = "/tmp/dd_ml_models_vector_details.csv"     # Linux

con.execute(f"""
COPY (
    SELECT DISTINCT
        modelDisplayName,
        modelVersionId,
        Data_Selection, 
        Features,
        count(distinct customerId) cnt_dstct_customer,
        count(distinct digitalLoanAccountId) cnt_dstct_loans,
        min(application_date) firstdate,
        max(application_date) lastdate
    FROM dd_ml_models_vector_details
    group by 1,2,3,4
    ORDER BY 1,2,3,4
) TO '{output_file}'
WITH (HEADER, DELIMITER ',');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""SELECT Features, count(distinct digitalLoanAccountId)cnt FROM dd_ml_models_vector_details 

group by 1
            """).df()

In [22]:
con.close()